# RAG local, passo a passo

Este notebook percorre, uma etapa por vez, o pipeline do `app.py`: carregar o PDF, dividir em trechos, gerar embeddings, indexar, recuperar e responder.

Ele usa as mesmas funções e configurações do `rag.py`, mas mostra os resultados intermediários que a interface esconde. Depois de cada etapa há um espaço para anotações.

**Antes de começar:**
- Rode o notebook a partir da pasta `notebooks/` (ele importa o `rag.py` da pasta acima).
- O repositório já traz `Data_science.pdf` como exemplo; `PDF_PATH` abaixo já aponta para ele. Para testar com outro PDF (com texto, não escaneado), troque o caminho.
- Para a etapa de resposta, o Ollama precisa estar rodando com o modelo baixado (`ollama pull granite4:micro`). As etapas anteriores funcionam sem ele.

## 0. Instalação e configuração

Rode uma vez por ambiente (pule se já tiver tudo pronto):

- **Dependências Python** - as mesmas do `rag.py`.
- **Ollama** - precisa estar instalado e com o modelo baixado antes da etapa 8 (geração).

In [ ]:
%pip install -q -r ../requirements.txt

In [ ]:
# Baixa o modelo padrão do projeto (sem efeito se já estiver baixado)
!ollama pull granite4:micro

## 1. Configuração

In [1]:
import sys
import textwrap
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))  # permite importar o rag.py

import rag

PDF_PATH = "../Data_science.pdf"  # exemplo incluído no repositório; troque para o seu PDF
PERGUNTA = "O que é Ciência de Dados?"  # ajuste para uma pergunta sobre o seu PDF


def imprimir_com_wrap(texto, largura=100):
    paragrafos = texto.split("\n\n")
    print("\n\n".join(textwrap.fill(p, width=largura) for p in paragrafos))


print("LLM:", rag.LLM_MODEL)
print("Embeddings:", rag.EMBEDDING_MODEL)
print("Chunk:", rag.CHUNK_SIZE, "caracteres | sobreposição:", rag.CHUNK_OVERLAP, "| top-k:", rag.TOP_K)
print("Temperatura:", rag.TEMPERATURE, "| máx. tokens por resposta:", rag.MAX_TOKENS)

LLM: granite4:micro
Embeddings: ibm-granite/granite-embedding-278m-multilingual
Chunk: 1000 caracteres | sobreposição: 150 | top-k: 4
Temperatura: 0.1 | máx. tokens por resposta: 350


> 📝 **Minhas anotações:**
>
> - Aqui os valores são os mesmos de `rag.py` → bom para validação do script.
> - E se eu quisesse variar os valores aqui para depois usar em `rag.py`?

## 2. Carregar o PDF

O `PyPDFLoader` cria um documento por página, com o texto e metadados como o número da página (contado a partir de zero).

In [2]:
from pprint import pprint

paginas = rag.load_pdf(PDF_PATH)
print(f"{len(paginas)} páginas")
print("Metadados da primeira página:")
pprint(paginas[0].metadata)
print()
imprimir_com_wrap(paginas[0].page_content[:500])

8 páginas
Metadados da primeira página:
{'creationdate': '2026-11-07T23:33:50+00:00',
 'creator': 'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like '
            'Gecko) HeadlessChrome/154.0.0.0 Safari/537.36',
 'moddate': '2026-11-07T23:33:50+00:00',
 'page': 0,
 'page_label': '1',
 'producer': 'Skia/PDF m154',
 'source': '../Data_science.pdf',
 'title': 'Data science - Wikipedia',
 'total_pages': 8}

The existence of Comet NEOWISE (here depicted as a series of red dots) was discovered by analyzing
astronomical survey data acquired by a space telescope, the Wide-field Infrared Survey Explorer.
Data science Data science is an interdisciplinary academic field[1] that uses statistics, scientific
computing, scientific methods, processing, scientific visualization, algorithms, coding (like
Python, SQL, and R), and systems to extract or extrapolate knowledge from potentially noisy,
structured, or u


Verifique se o texto saiu legível. PDFs escaneados chegam vazios aqui, e nenhuma etapa seguinte consegue corrigir isso.

> 📝 **Minhas anotações:**
>
> Ok.

## 3. Dividir em trechos (chunking)

O `RecursiveCharacterTextSplitter` tenta cortar em parágrafos, depois em linhas, depois em palavras, até cada trecho caber no tamanho definido. A sobreposição repete um pedaço do fim de um trecho no começo do seguinte, para não perder frases cortadas ao meio.

In [3]:
trechos = rag.split(paginas)
tamanhos = [len(t.page_content) for t in trechos]

print(f"{len(trechos)} trechos")
print(f"Tamanho: mínimo {min(tamanhos)}, médio {sum(tamanhos) // len(tamanhos)}, máximo {max(tamanhos)} caracteres")
print()
for t in trechos[:2]:
    print(f"--- página {t.metadata['page'] + 1} ---")
    imprimir_com_wrap(t.page_content[:300])
    print("...\n")

31 trechos
Tamanho: mínimo 442, médio 872, máximo 996 caracteres

--- página 1 ---
The existence of Comet NEOWISE (here depicted as a series of red dots) was discovered by analyzing
astronomical survey data acquired by a space telescope, the Wide-field Infrared Survey Explorer.
Data science Data science is an interdisciplinary academic field[1] that uses statistics, scientific
com
...

--- página 1 ---
domain knowledge from the underlying application domain (e.g., natural sciences, information
technology, and medicine).[5] Data science is multifaceted and can be described as a science, a
research paradigm, a research method, a discipline, a workflow, and a profession.[6] Data science is
"a concept
...



**Experimento:** mude o tamanho do trecho e compare quantos trechos saem e onde os cortes caem.

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

for tamanho in (300, 1000, 2000):
    splitter = RecursiveCharacterTextSplitter(chunk_size=tamanho, chunk_overlap=tamanho // 7)
    print(f"chunk_size={tamanho}: {len(splitter.split_documents(paginas))} trechos")

chunk_size=300: 101 trechos
chunk_size=1000: 31 trechos
chunk_size=2000: 19 trechos


> 📝 **Minhas anotações:**
>
> Ok.

## 4. Embeddings

Cada trecho vira um vetor de números. Textos com significado parecido ficam próximos nesse espaço. Na primeira execução, o modelo é baixado do Hugging Face; depois é lido do cache local.

In [5]:
emb = rag.get_embeddings()

vetor = emb.embed_query(PERGUNTA)
print("Dimensão do vetor:", len(vetor))
print("Primeiros valores:", [round(float(v), 4) for v in vetor[:5]])

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

This model was loaded in torch.bfloat16 on CPU. Depending on your CPU, inference may be significantly slower than with torch.float32. Consider comparing inference speeds by loading with `model_kwargs={"dtype": torch.float32}`.


Dimensão do vetor: 768
Primeiros valores: [-0.0649, 0.0111, 0.0137, 0.0149, 0.0554]


**Experimento:** como os vetores estão normalizados, o produto escalar é a similaridade de cosseno. Frases com o mesmo sentido devem ter similaridade maior, mesmo com palavras diferentes ou em outro idioma.

In [6]:
import numpy as np

frases = [
    "O titular pode pedir acesso aos seus dados.",
    "A pessoa tem direito de consultar as informações que a empresa guarda sobre ela.",
    "The data subject may request access to their data.",
    "A receita do bolo leva três ovos.",
]
vetores = np.array(emb.embed_documents(frases))
similaridade = vetores @ vetores.T

for i, frase in enumerate(frases[1:], start=1):
    print(f"{similaridade[0, i]:.3f}  {frase}")

0.751  A pessoa tem direito de consultar as informações que a empresa guarda sobre ela.
0.935  The data subject may request access to their data.
0.399  A receita do bolo leva três ovos.


> 📝 **Minhas anotações:**
>
> Similaridade: cosseno entre dois vetores.
>
> - cos(0) = 1 → ângulo entre os vetores é zero → são iguais.
> - cos(π/2) = 0 → ângulo entre os vetores é 90° → são diferentes.

## 5. Indexar no Chroma

O índice é criado uma vez por PDF (identificado pelo hash do conteúdo) e salvo em disco. Rodar esta célula de novo reaproveita o índice em vez de recalcular os embeddings.

In [7]:
digest = rag.file_hash(PDF_PATH)
store = rag.get_vectorstore(PDF_PATH, digest)
print("Hash do PDF:", digest)
print("Índice em:", rag.INDEX_DIR / digest)

Hash do PDF: fd8a07e4d565235d
Índice em: .chroma/fd8a07e4d565235d


> 📝 **Minhas anotações:**
>
> - O hash com base no conteúdo é uma forma de garantir que o mesmo documento não será indexado com multiplicidade.
> - Conteúdos iguais mas em meios diferentes (PDF e página HTML com o mesmo texto) provavelmente não terão o mesmo hash.
> - Se os documentos são persistidos, como apagar a base vetorial?
> - Tem cache (LRU) pra 16 docs.

## 6. Recuperar os trechos mais relevantes

Antes de envolver o LLM, vale olhar o que a busca devolve. Se os trechos certos não aparecem aqui, o LLM não tem como acertar: é uma falha de recuperação, não de geração.

In [8]:
resultados = store.similarity_search_with_score(PERGUNTA, k=rag.TOP_K)

for i, (doc, distancia) in enumerate(resultados, start=1):
    print(f"C{i} | página {doc.metadata['page'] + 1} | distância {distancia:.3f}")
    imprimir_com_wrap(doc.page_content[:250].replace("\n", " "))
    print("...\n")

C1 | página 1 | distância 0.246
domain knowledge from the underlying application domain (e.g., natural sciences, information
technology, and medicine).[5] Data science is multifaceted and can be described as a science, a
research paradigm, a research method, a discipline, a workflo
...

C2 | página 2 | distância 0.291
data-science program. He describes data science as an applied field growing out of traditional
statistics.[16] In 1962, John Tukey described a field he called "data analysis", which resembles
modern data science.[16] In 1985, in a lecture given to th
...

C3 | página 1 | distância 0.303
because of the impact of information technology" and the data deluge.[9][10] Data science is often
described as a multidisciplinary and rapidly evolving field because it draws on techniques from
diverse areas, such as computer science, statistics, in
...

C4 | página 2 | distância 0.322
proposed using the term ‘data science’ rather than ‘computer science’ to reflect the growing
emphasis on

No Chroma, o valor exibido é uma **distância**: quanto menor, mais parecido.

> 📝 **Minhas anotações:**
>
> - k = 4 parece ok, mas variar o k pode dar resultados diferentes (testar).

## 7. Montar o prompt

O contexto entra numerado, com a página, para que o LLM possa citar as fontes. As instruções pedem que ele use só esse contexto e admita quando a resposta não está no documento.

In [9]:
docs = [doc for doc, _ in resultados]
contexto = rag.format_context(docs)
mensagens = rag.PROMPT.invoke({"context": contexto, "question": PERGUNTA})

for m in mensagens.to_messages():
    print(f"=== {m.type} ===")
    imprimir_com_wrap(m.content)
    print()

=== system ===
Você é um assistente que responde perguntas sobre um documento. Use APENAS os trechos fornecidos no
contexto. Cite os trechos que sustentam a resposta pelo rótulo, ex.: (C1). Não use colchetes para
citar: o documento pode ter suas próprias notas numeradas entre colchetes, que não têm relação com
os trechos do contexto. Se o contexto não contiver a resposta, diga claramente que não encontrou a
informação no documento, sem inventar. Responda no mesmo idioma da pergunta.

=== human ===
Contexto: C1 (p. 1): domain knowledge from the underlying application domain (e.g., natural
sciences, information technology, and medicine).[5] Data science is multifaceted and can be
described as a science, a research paradigm, a research method, a discipline, a workflow, and a
profession.[6] Data science is "a concept to unify statistics, data analysis, informatics, and their
related methods" to "understand and analyze actual phenomena" with data.[7] It uses techniques and
theories drawn fr

> 📝 **Minhas anotações:**
>
> - A saída de 1500 caracteres não dava pra ver o fim (pergunta) → retirou o limite (output truncado, mas a pergunta está visível).

## 8. Gerar a resposta

Esta etapa precisa do Ollama rodando. Em CPU, pode levar de alguns segundos a algumas dezenas de segundos.

In [10]:
import time

try:
    inicio = time.perf_counter()
    resposta = rag.get_llm().invoke(mensagens)
    print(f"({time.perf_counter() - inicio:.1f} s)\n")
    imprimir_com_wrap(resposta.content)
except Exception as erro:
    print("Não foi possível chamar o LLM. O Ollama está rodando e o modelo foi baixado?")
    print(erro)

(85.5 s)

Ciência de Dados é um campo multidisciplinar que se concentra na extração de conhecimento a partir
de grandes conjuntos de dados e na aplicação desse conhecimento para resolver problemas em outros
domínios de aplicação. É uma área que combina técnicas de diversas áreas, como ciência da
computação, estatística, ciência da informação e outras disciplinas específicas. A Ciência de Dados
é frequentemente descrita como uma disciplina em rápida evolução e em constante mudança devido à sua
natureza interdisciplinar e à crescente importância da informação tecnológica e dos grandes volumes
de dados.


Confira a resposta contra os trechos da etapa 6: cada afirmação aparece neles? As citações (Cn) apontam para o trecho certo?

> 📝 **Minhas anotações:**
>
> - Onde estão os trechos citados?

## 9. Teste de recusa

Uma pergunta que o documento não responde. O esperado é o modelo dizer que não encontrou a informação, em vez de inventar.

In [11]:
pergunta_fora = "Qual é a capital da Austrália?"
try:
    resposta, _contexto, t_recuperacao, t_geracao, t_total = rag.answer(PDF_PATH, pergunta_fora)
    imprimir_com_wrap(resposta)
    print(f"\n(recuperação {t_recuperacao}s | geração {t_geracao}s | total {t_total}s)")
except Exception as erro:
    print("Não foi possível chamar o LLM:", erro)

A pergunta não está relacionada ao contexto fornecido. Não há informação sobre a capital da
Austrália no texto.

(recuperação 2.8s | geração 38.3s | total 41.2s)


> 📝 **Minhas anotações:**
>
> - É preciso fazer testes limítrofes (falar de algo no assunto, mas que não tem o dado em si no texto).

## 10. Limpeza (apagar o índice)

Remove o índice do PDF atual do disco e do cache em memória (`lru_cache`). Útil para forçar reindexação, por exemplo depois de mudar `CHUNK_SIZE` ou o modelo de embedding.

In [12]:
import shutil

shutil.rmtree(rag.INDEX_DIR / digest, ignore_errors=True)
rag.get_vectorstore.cache_clear()  # sem isso, o objeto em memória continuaria apontando pra pasta apagada
print("Índice removido:", rag.INDEX_DIR / digest)

Índice removido: .chroma/fd8a07e4d565235d


## Conclusões

- O que funcionou bem: chunks, geração de texto.
- Onde o sistema errou, e em qual etapa (recuperação ou geração): resposta não trouxe citação (é pedido no prompt e funcionou em rag.py).
- O que eu mudaria na próxima versão: usar modelo usando API token (velocidade), usar modelos com e sem reasoning, melhorar orquestração, testar novas estratégias de chunking, variar documentos, fazer perguntas limítrofes, anotar métricas de qualidade do output.